# Syon 3 — Treinamento DO ZERO (Kaggle)

**Arquitetura Syon 3 proprietária** — tokenizer BPE + pesos aleatórios + 3 fases

**Settings:** GPU T4 + Internet ON

> **Ordem:** célula 1 (setup) → GPU → pip → **célula 4 (treino do zero)**
> Modelo 100% Syon. Sem TinyLlama, LoRA ou modelos de terceiros.

In [ ]:
import os, sys, shutil
from pathlib import Path

INPUT = Path("/kaggle/input")
PROJECT = Path("/kaggle/working/syon")

print("=== /kaggle/input ===")
if INPUT.exists():
    for item in sorted(INPUT.iterdir()):
        print(" ", item.name + ("/" if item.is_dir() else ""))
else:
    print("  (vazio — adicione o Dataset no painel direito)")

# Busca automatica do projeto Syon 3
SYON_SRC = None
candidates = [
    Path("/kaggle/input/datasets/regyfelipe/syon-project/Syon"),
    Path("/kaggle/input/syon-project/Syon"),
    Path("/kaggle/input/models/regyfelipe/syon-3/pytorch/default/1/Syon"),
    Path("/kaggle/input/models/regyfelipe/syon-3/pytorch/default/Syon"),
]
for c in candidates:
    if c.is_dir() and (c / "models/architecture/syon3.py").exists():
        SYON_SRC = c
        break

if SYON_SRC is None:
    for marker in INPUT.rglob("syon3.py"):
        root = marker.parent.parent.parent
        if (root / "training").is_dir():
            SYON_SRC = root
            print("Encontrado via busca:", SYON_SRC)
            break

if SYON_SRC is None:
    raise FileNotFoundError(
        "Syon 3 nao encontrado.\n"
        "1) Painel direito → Add Data → Your Datasets → syon-project\n"
        "2) Re-envie o dataset com a pasta Syon/ (com syon3.py)\n"
        "3) Rode esta celula de novo"
    )

if PROJECT.exists():
    shutil.rmtree(PROJECT)
shutil.copytree(SYON_SRC, PROJECT)
os.chdir(PROJECT)
sys.path[:0] = [str(PROJECT), str(PROJECT / "src")]

print("Origem:", SYON_SRC)
print("cwd:", os.getcwd())
print("syon3.py:", (PROJECT / "models/architecture/syon3.py").exists())
print("\n✓ Setup OK — continue celulas 2 → 3 → 4.")

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
import os
assert os.getcwd().endswith("syon"), "ERRO: rode a celula 1 antes! cwd=" + os.getcwd()
!pip install -q pyyaml pydantic-settings safetensors

In [ ]:
import os, subprocess, sys
from pathlib import Path

PROJECT = Path("/kaggle/working/syon")
os.chdir(PROJECT)
DATA = PROJECT / "data/raw"
CONFIG = PROJECT / "training/configs/syon_scratch_kaggle.yaml"
PRETRAIN = PROJECT / "training/checkpoints/pretrain/best"

def run(cmd):
    env = os.environ.copy()
    paths = [str(PROJECT), str(PROJECT / "src")]
    if extra := env.get("PYTHONPATH"):
        paths.append(extra)
    env["PYTHONPATH"] = os.pathsep.join(paths)
    print("\n>>", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(PROJECT), env=env)

steps = [
    [sys.executable, "scripts/data/build_master_curriculum.py", "--output", str(DATA), "--min-samples", "5000"],
    [sys.executable, "scripts/data/process_pipeline.py", "--raw", str(DATA)],
    [sys.executable, "scripts/data/train_syon_tokenizer.py", "--data-dir", str(DATA),
     "--output", str(PROJECT / "models/tokenizer/syon3-bpe")],
    [sys.executable, "-m", "training.pretrain", "--config", str(CONFIG), "--data-dir", str(DATA)],
    [sys.executable, "-m", "training.master_trainer", "--config", str(CONFIG), "--phase", "1",
     "--data-dir", str(DATA), "--resume", str(PRETRAIN)],
    [sys.executable, "-m", "training.master_trainer", "--config", str(CONFIG), "--phase", "2",
     "--data-dir", str(DATA), "--resume", str(PROJECT / "training/checkpoints/phase1/best")],
    [sys.executable, "-m", "training.master_trainer", "--config", str(CONFIG), "--phase", "3",
     "--data-dir", str(DATA), "--resume", str(PROJECT / "training/checkpoints/phase2/best")],
]

for i, cmd in enumerate(steps, 1):
    print(f"\n{'='*50}\n[DO ZERO] PASSO {i}/{len(steps)}\n{'='*50}")
    run(cmd)

print("\n✓ SYON 3 → /kaggle/working/syon-output/syon-3")

## Publicar no Kaggle Models (após treino)

Rode a célula abaixo **só depois** do passo 7 concluir. Handle: `regyfelipe/syon-3/pytorch/default`

In [ ]:
import json, shutil, sys
from datetime import datetime, timezone
from pathlib import Path

SOURCE = Path("/kaggle/working/syon-output/syon-3")
OUT = Path("/kaggle/working/dist/kaggle-model/syon-3")
HANDLE = "regyfelipe/syon-3/pytorch/default"

if not SOURCE.exists():
    raise FileNotFoundError(f"Modelo nao encontrado: {SOURCE}. Treine de novo ou restaure output.")

for f in ("config.json", "pytorch_model.bin", "tokenizer.json"):
    if not (SOURCE / f).exists():
        raise FileNotFoundError(f"Falta {f} em {SOURCE}")

if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)
for f in ("config.json", "pytorch_model.bin", "tokenizer.json", "syon_model.json"):
    src = SOURCE / f
    if src.exists():
        shutil.copy2(src, OUT / f)

card = {
    "model_name": "Syon 3",
    "variation": "default",
    "from_scratch": True,
    "packaged_at": datetime.now(timezone.utc).isoformat(),
}
(OUT / "kaggle_model_card.json").write_text(json.dumps(card, indent=2))

!pip install -q kagglehub
import kagglehub
kagglehub.model_upload(HANDLE, str(OUT), version_notes="Syon 3 — treinado do zero v1")
print("✓ Publicado:", HANDLE)

In [ ]:
import json
from pathlib import Path
s = Path("/kaggle/working/syon-output/syon-3_summary.json")
if s.exists():
    print(json.dumps(json.loads(s.read_text()), indent=2))
model = Path("/kaggle/working/syon-output/syon-3")
print("Syon 3:", model.exists(), "| arquivos:", list(model.glob("*")) if model.exists() else [])

## Inferência Syon 3 (cole no Kaggle — não depende de arquivos locais)

**Input:** Model `syon-3` V4 (pesos) + V1 (código) ou notebook output

In [ ]:
# Inferencia — so precisa Model syon-3 V4 (+ notebook1 output OU V1 como codigo)
!pip install -q kagglehub
import sys, shutil
from pathlib import Path
import torch
import kagglehub

INPUT = Path("/kaggle/input")

def find_code():
    for marker in INPUT.rglob("syon3.py"):
        root = marker.parent.parent.parent
        if (root / "training").is_dir():
            return root
    for name in ("syon", "Syon"):
        nb = INPUT / "notebooks" / "regyfelipe" / "notebook1" / name
        if (nb / "models/architecture/syon3.py").exists():
            return nb
    cache = Path(kagglehub.notebook_output_download("regyfelipe/notebook1/versions/2"))
    for marker in cache.rglob("syon3.py"):
        root = marker.parent.parent.parent
        if (root / "training").is_dir():
            return root
    return None

def find_weights():
    cands = [d for cfg in INPUT.rglob("config.json")
             for d in [cfg.parent]
             if (d/"pytorch_model.bin").exists() and (d/"tokenizer.json").exists()]
    return max(cands, key=lambda p: int([x for x in p.parts if x.isdigit()][-1]))

CODE = find_code()
assert CODE, "Add Input: notebook1 V2 output OU syon-3 V1"
WEIGHTS = find_weights()
for p in (str(CODE), str(CODE / "src")):
    sys.path.insert(0, p)
print("Codigo:", CODE)
print("Pesos:", WEIGHTS)

from models.architecture.syon3 import Syon3
from models.tokenizer.syon_bpe import SyonBPETokenizer

model = Syon3.from_pretrained(WEIGHTS, device="cuda")
tokenizer = SyonBPETokenizer.from_pretrained(WEIGHTS)
model.eval()

prompt = "<|user|> O que é SQL injection?\n<|assistant|>"
ids = tokenizer.encode(prompt, add_special=False)
x = torch.tensor([ids], device="cuda")
with torch.no_grad():
    for _ in range(80):
        logits = model(x).logits[:, -1, :]
        x = torch.cat([x, logits.argmax(dim=-1, keepdim=True)], dim=1)
print(tokenizer.decode(x[0].tolist()))